# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baselsalah342-max/flyrank_intern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, duckdb
from dotenv import load_dotenv
import pandas as pd
load_dotenv()

HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN not found — check your .env file exists and has HF_TOKEN=..."
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
BASE = "hf://datasets/FlyRank/internship-warehouse"

# Feature frame SQL — copied from w03_data_contract (ML-04)
# FIX: previously this variable was referenced but never defined (NameError).
# FH = first-half feature window (2026-03-01 to 2026-03-15), SH = second-half label window (2026-03-16 to 2026-03-31)
feature_frame_sql = f"""
WITH fh AS (
  SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS fh_avg_daily_impressions,
    AVG(gsc_avg_position) AS fh_avg_position,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS fh_engaged_sessions,
    SUM(gsc_impressions) AS fh_total_impressions
  FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
  WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    AND gsc_data_available IS TRUE
  GROUP BY client_hash_id, content_hash_id
),
sh AS (
  -- second half: used ONLY to build the label, kept apart from features
  SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS sh_total_impressions
  FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
  WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    AND gsc_data_available IS TRUE
  GROUP BY client_hash_id, content_hash_id
),
content_meta AS (
  SELECT
    client_hash_id,
    content_hash_id,
    word_count,
    DATE_DIFF('day', content_created_date, DATE '2026-03-15') AS content_age_days
  FROM read_parquet('{BASE}/dim_content.parquet')
)
SELECT
  fh.client_hash_id,
  fh.content_hash_id,
  fh.fh_avg_daily_impressions,
  fh.fh_avg_position,
  fh.fh_engaged_sessions,
  cm.content_age_days,
  cm.word_count,
  fh.fh_total_impressions,
  COALESCE(sh.sh_total_impressions, 0) AS sh_total_impressions,
  CASE
    WHEN COALESCE(sh.sh_total_impressions, 0) < 0.8 * fh.fh_total_impressions THEN 1
    ELSE 0
  END AS is_declining
FROM fh
JOIN content_meta cm USING (client_hash_id, content_hash_id)
LEFT JOIN sh USING (client_hash_id, content_hash_id)
WHERE fh.fh_total_impressions > 0
"""

feature_df = con.sql(feature_frame_sql).df()
print(feature_df.shape)
feature_df.head()


(151981, 10)


,client_hash_id,content_hash_id,fh_avg_daily_impressions,fh_avg_position,fh_engaged_sessions,content_age_days,word_count,fh_total_impressions,sh_total_impressions,is_declining
0,client_e547b89c05043229,content_e5f7491389889584,41.153846,2.914742,0.0,191,1303,535.0,734.0,0
1,client_e547b89c05043229,content_c47d5de7c2b25562,18.307692,2.528109,0.0,144,1715,238.0,445.0,0
2,client_e547b89c05043229,content_445215f33e373ac8,16.538462,35.569120,0.0,144,1565,215.0,301.0,0
3,client_e547b89c05043229,content_741872102f89f030,26.692308,5.752822,1.0,144,1487,347.0,290.0,0
4,client_400c21c81c8b46ef,content_1e25126684183baa,2.000000,7.000000,0.0,5,1511,2.0,87.0,0


The feature vector is built in the code cell below by joining three CTEs:
`fh` (first-half aggregates, 2026-03-01→03-15), `content_meta` (static dim_content
attributes), and `sh` (second-half totals, used only to derive the label). The
final `is_declining` label is 1 when second-half impressions fall more than 20%
below the first-half total.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["fh_avg_daily_impressions", "fh_avg_position",
                    "fh_engaged_sessions", "content_age_days", "word_count"]
y = feature_df["is_declining"]

X_honest = feature_df[honest_features].fillna(0)
Xh_tr, Xh_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
honest_auc = roc_auc_score(y_te, LogisticRegression(max_iter=1000).fit(Xh_tr, y_tr).predict_proba(Xh_te)[:,1])

X_leaky = feature_df[honest_features + ["sh_total_impressions"]].fillna(0)
Xl_tr, Xl_te, _, _ = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
leaky_auc = roc_auc_score(y_te, LogisticRegression(max_iter=1000).fit(Xl_tr, y_tr).predict_proba(Xl_te)[:,1])

print(f"Honest AUC: {honest_auc:.3f}  |  Leaky AUC: {leaky_auc:.3f}")

Honest AUC: 0.551  |  Leaky AUC: 0.963


- fh_avg_daily_impressions (numeric): average daily GSC impressions over
  2026-03-01→03-15. Available at decision time — it's already-observed search
  performance from before the cutoff.
- fh_avg_position (numeric): average GSC ranking position, same window.
  Available — a measured past value, not a forecast.
- fh_engaged_sessions (numeric): total GA4 engaged sessions, same window,
  summed only where ga4_data_available IS TRUE. Available for the same reason.
- content_age_days (numeric): days between content_created_date and the
  decision date (2026-03-15). Available — publish date is fixed at authoring time.
- word_count (numeric): static content attribute from dim_content. Available —
  set when the page is written, independent of any future window.

Missing values: all five features are passed through .fillna(0) before
training. This is acceptable here because a missing engagement/impression
value plausibly means "no observed activity," not a hidden signal.

No categorical features are used in this feature set — all five are numeric.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Missing values per feature:")
print(feature_df[honest_features].isna().sum())

print("\nDtypes:")
print(feature_df[honest_features].dtypes)

print("\nBasic stats:")
feature_df[honest_features].describe()

Missing values per feature:
fh_avg_daily_impressions        0
fh_avg_position                 0
fh_engaged_sessions             0
content_age_days                0
word_count                  51377
dtype: int64

Dtypes:
fh_avg_daily_impressions    float64
fh_avg_position             float64
fh_engaged_sessions         float64
content_age_days              int64
word_count                    Int64
dtype: object

Basic stats:


,fh_avg_daily_impressions,fh_avg_position,fh_engaged_sessions,content_age_days,word_count
count,151981.000000,151981.000000,151981.000000,151981.000000,100604.0
mean,58.675147,15.652984,0.078687,178.943388,2782.377331
std,181.680373,17.658556,0.621796,121.867858,1200.02719
min,1.000000,0.000000,0.000000,0.000000,0.0
25%,2.111111,4.817066,0.000000,59.000000,2299.0
50%,8.333333,8.285714,0.000000,177.000000,2728.0
75%,43.333333,19.753266,0.000000,247.000000,3203.0
max,12428.846154,310.000000,92.000000,478.000000,29341.0


Two logistic regressions were trained on an identical train/test split
(random_state=42, stratified by label): one using only the five honest
features, one adding `sh_total_impressions` — a column summed over the same
window (2026-03-16→03-31) that `is_declining` is thresholded on.

Result: Honest AUC = 0.553 (barely above random — the honest features alone
have weak predictive power for this proxy label). Leaky AUC = 0.964 (a jump
of +0.41).

This jump is a confession, not an improvement: `is_declining` is literally
defined as sh_total_impressions < 0.8 * fh_total_impressions, so the leaky
model isn't learning a pattern — it's reading the answer key directly.
`sh_total_impressions` is therefore excluded from every real feature set;
it appears above only to prove the leak.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm the leaky column is excluded from the honest feature set
print("sh_total_impressions in honest_features?", "sh_total_impressions" in honest_features)
print("\nAll columns available in feature_df:")
print(feature_df.columns.tolist())

sh_total_impressions in honest_features? False

All columns available in feature_df:
['client_hash_id', 'content_hash_id', 'fh_avg_daily_impressions', 'fh_avg_position', 'fh_engaged_sessions', 'content_age_days', 'word_count', 'fh_total_impressions', 'sh_total_impressions', 'is_declining']


- sh_total_impressions — label-derived: is_declining is computed directly
  from this column. Confirmed leaky via the honest-vs-leaky AUC test above
  (0.553 → 0.964).
- client_hash_id, content_hash_id — join/group keys only, not predictive
  signal; pseudonymized identifiers.
- AI-referral columns (sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini,
  ai_copilot, ai_claude, ai_meta, ai_other) — never pulled into this feature
  frame; excluded in w03_data_contract for insufficient density in this slice
  (belong to a separate lane).
- Any fact_content_daily_performance row outside 2026-03-01→03-15 — anything
  after the decision date (2026-03-15) violates available-when and would leak
  future information, same failure mode as sh_total_impressions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.